# PointProbability

The PointProbability class implements probabilistic classification methods.\
It can be used to estimate the probability that a test galaxy is star-forming, or that it has a stellar mass or redshift in a given range of values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fomoz.data import Dataset, SelectionDefinition
from fomoz.probability import PointProbability

## Input samples

For demonstration purposes, we use the same catalog as both the reference and test samples. This illustrates the workflow, not performance on an independent sample. In realistic scenarios, reference and test samples must be independent.

Magnitude errors must be present in the input catalogs, because they define the matching between reference and test objects. Here, we assume 0.1 mag errors.

In [ ]:
# Load the sample and assign dummy magnitude errors
sample = pd.read_csv('../data/input/reference.csv', comment='#')
bands = ['g', 'r', 'i']

for band in bands:
    if f'{band}_error' not in sample:
        sample[f'{band}_error'] = 0.1
        print(f'{band}: assuming 0.1 mag uncertainty for this example.')

# Dataset
reference = Dataset(sample, bands, z_gal_string='redshift')

## Selection probabilities
Estimate the probability that each test galaxy lies above a redshift threshold.

In [ ]:
# Targeted sample
z_limit = 0.6
selection = SelectionDefinition(z_lim=z_limit)

# --- Probabilistic classification using PointProbability ---
point = PointProbability(
    selection_definition=selection,
    reference_dataset=reference,
    test_dataset=reference,  # in a realistic scenario, this should be an actual test sample (see above)
)

point.select_features_grid()  # select the most informative colors
point.process(cpu_number=1)  # process

# Get the original DataFrame, which also includes the following keys:
# fomoz_prob: Weighted membership probability.
# fomoz_prob_error: Approximate sampling standard error of the probability.
# fomoz_n_good: Sum of weighted memberships over matching references, i.e. the numerator in fomoz_prob.
# fomoz_n_all: Unweighted number of matching reference objects.
# fomoz_n_colors_used: Number of usable selected colors for each test object.
results = point.get_results_dataframe()

In [ ]:
print("\nReminder: this example is based only on 3 photometric bands (g, r, i) and a small reference sample\n")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(sample['redshift'], results['fomoz_prob'], s=15, alpha=0.6)
ax.axvline(z_limit, color='black', linestyle='--', label='Selection threshold')
ax.set_xlabel('Catalog redshift', fontsize=14)
ax.set_ylabel(r'$P\,(z > z_{\rm lim})$', fontsize=14)
ax.legend()
plt.show()